<a href="https://colab.research.google.com/github/DEEPTHIVM/MD-simulation-OpenMM/blob/main/MD_simulation_(OpenMM).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install Conda support in Colab

In [ ]:
!pip -q install condacolab
import condacolab; condacolab.install()


The next command installs OpenMM, PDBFixer, MDTraj, RDKit, AmberTools, Matplotlib, and pandas from conda-forge. These packages support structure preparation, ligand parameterization, simulation, and analysis.

In [ ]:
!mamba install -q -y -c conda-forge openmm pdbfixer mdtraj rdkit ambertools matplotlib pandas

This section mounts Google Drive, creates the working directory, and loads the docked complex if it is not already present. The SMILES string describes the ligand whose chemical identity will be used during preparation.

In [ ]:
from google.colab import drive, files
import os
drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/md', exist_ok=True)
os.chdir('/content/drive/MyDrive/md')
files.upload()   # only if docked.pdb is not already in this folder
SMILES = "C[C@]12CC[C@H]3C(=CC(=O)[C@H]4[C@@]3(C[C@@H]([C@@H](C4)O)O)C)[C@@]1(CC[C@@H]2[C@](C)([C@@H](CCC(C)(C)O)O)O)O"

The code defines the ligand residue name and standard amino-acid residue names, reads the complex structure, and writes separate protein and ligand PDB files. The element mapping is used while cleaning atom records.

In [ ]:
LIG = "UNL"
AA = {"ALA","ARG","ASN","ASP","CYS","GLN","GLU","GLY","HIS","ILE","LEU","LYS","MET","PHE","PRO","SER","THR","TRP","TYR","VAL"}
amap = {"OA":"O","NA":"N","SA":"S","HD":"H","HS":"H","A":"C"}
L = open("complex.pdb").read().split("ENDMDL")[0].splitlines()

def clean(lines, out):
    o = []
    for l in lines:
        t = l.split()[-1].upper()
        el = amap.get(t, t.capitalize())
        if el == "H": continue
        o.append(l[:54] + "  1.00  0.00" + " "*10 + el.rjust(2))
    open(out, "w").write("\n".join(o) + "\nEND\n")
    print(out, len(o), "heavy atoms")

clean([l for l in L if l.startswith(("ATOM","HETATM")) and l[17:20].strip() in AA], "protein.pdb")
clean([l for l in L if l.startswith(("ATOM","HETATM")) and l[17:20].strip() == LIG], "ligand.pdb")

RDKit reads the ligand coordinates and the supplied SMILES template, assigns bond orders, adds hydrogen atoms, and writes an SDF file. The formal charge is recorded for the subsequent AmberTools parameterization step.

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
tmpl = Chem.MolFromSmiles(SMILES)
lig = Chem.MolFromPDBFile("ligand.pdb", removeHs=True)
lig = AllChem.AssignBondOrdersFromTemplate(tmpl, lig)
lig = Chem.AddHs(lig, addCoords=True)
Chem.MolToMolFile(lig, "ligand.sdf")
NET_CHARGE = Chem.GetFormalCharge(lig); print("net charge:", NET_CHARGE)

PDBFixer loads the protein-only structure, checks for missing residues and atoms, and writes a prepared protein structure. Missing-residue insertion is explicitly disabled in the existing code.

In [ ]:
from pdbfixer import PDBFixer
from openmm.app import PDBFile
fx = PDBFixer("protein.pdb")
fx.findMissingResidues(); fx.missingResidues = {}
fx.findMissingAtoms(); fx.addMissingAtoms()
PDBFile.writeFile(fx.topology, fx.positions, open("prot_fixed.pdb","w"), keepIds=True)

AmberTools Antechamber assigns GAFF2 atom types and AM1-BCC charges using the ligand charge calculated above. Parmchk2 then writes any additional force-field parameters needed for system construction.

In [ ]:
!antechamber -i ligand.sdf -fi sdf -o lig.mol2 -fo mol2 -c bcc -nc {NET_CHARGE} -at gaff2 -rn LIG -s 0
!parmchk2 -i lig.mol2 -f mol2 -o lig.frcmod -s gaff2
!ls -la lig.mol2 lig.frcmod

The LEaP input loads the protein and ligand force fields, combines the prepared structures, adds a TIP3P solvent box, and adds sodium and chloride ions before writing Amber topology and coordinate files.

In [ ]:
open("tleap.in","w").write("""
source leaprc.protein.ff14SB
source leaprc.gaff2
source leaprc.water.tip3p
loadamberparams lig.frcmod
LIG = loadmol2 lig.mol2
P = loadpdb prot_fixed.pdb
cplx = combine {P LIG}
solvateBox cplx TIP3PBOX 10.0
addIons2 cplx Na+ 0
addIons2 cplx Cl- 0
saveamberparm cplx system.prmtop system.inpcrd
quit
""")
!rm -f system.prmtop system.inpcrd
!tleap -f tleap.in | grep -E "FATAL|ERROR|Total unperturbed|Adding|Exiting"
!ls -la system.prmtop system.inpcrd

This cell writes a helper module that loads the Amber system, creates the OpenMM system and barostat, configures a Langevin-middle integrator, selects the CUDA platform, and sets the initial coordinates and periodic box.

In [ ]:
open("common.py","w").write('''
from openmm.app import *
from openmm import *
from openmm.unit import *
def make_sim():
    prmtop = AmberPrmtopFile("system.prmtop"); inpcrd = AmberInpcrdFile("system.inpcrd")
    system = prmtop.createSystem(nonbondedMethod=PME, nonbondedCutoff=1*nanometer,
                                 constraints=HBonds, hydrogenMass=1.5*amu)
    system.addForce(MonteCarloBarostat(1*bar, 300*kelvin))
    integ = LangevinMiddleIntegrator(300*kelvin, 1/picosecond, 0.004*picoseconds)
    sim = Simulation(prmtop.topology, system, integ, Platform.getPlatformByName("CUDA"), {"Precision":"mixed"})
    sim.context.setPositions(inpcrd.positions)
    if inpcrd.boxVectors is not None: sim.context.setPeriodicBoxVectors(*inpcrd.boxVectors)
    return sim, prmtop, inpcrd
''')

The NVIDIA utility output is used to inspect the GPU visible to the Colab runtime and its driver information.

In [ ]:
!nvidia-smi | head -15

Check OpenMM installation and platforms

In [ ]:
import openmm
print(openmm.__version__)
print([openmm.Platform.getPlatform(i).getName() for i in range(openmm.Platform.getNumPlatforms())])

The working directory is restored and the topology, coordinate, and helper files are checked before continuing with simulation setup.

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/md')
!ls system.prmtop system.inpcrd common.py

Inspect installed OpenMM and CUDA packages

In [ ]:
!mamba list | grep -E "openmm|cuda-version|cuda-nvrtc"

Install a CUDA 12-compatible OpenMM build

In [ ]:
!mamba install -y -c conda-forge "openmm=*=*cuda12*" "cuda-version=12.6" "cuda-nvrtc=12.6.*"

The command removes a CUDA-version pin from Conda's pinned configuration and displays the resulting file. This changes the runtime's package-resolution settings.

In [ ]:
!sed -i '/cuda-version/d' /usr/local/conda-meta/pinned
!cat /usr/local/conda-meta/pinned

Install OpenMM with the requested CUDA version

In [ ]:
!mamba install -y -c conda-forge openmm "cuda-version=12.6"

Run OpenMM's installation test

In [ ]:
!python -m openmm.testInstallation

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/md')

In [ ]:
!ls system.prmtop system.inpcrd common.py

This cell writes an equilibration script that loads the system, saves a PDB snapshot, minimizes the energy, assigns velocities at 300 K, runs the configured equilibration steps, resets the simulation clock and step counter, and saves an equilibration state.

In [ ]:
open("equil.py","w").write('''
from common import *
sim, prmtop, inpcrd = make_sim()
PDBFile.writeFile(prmtop.topology, inpcrd.positions, open("system.pdb","w"))
print(prmtop.topology.getNumAtoms(), "atoms", flush=True)
sim.minimizeEnergy()
sim.context.setVelocitiesToTemperature(300*kelvin)
sim.step(250000)
sim.context.setTime(0); sim.context.setStepCount(0)
sim.saveState("equil.xml"); print("equilibration done")
''')

Run equilibration

In [ ]:
!python equil.py

Prepare the production MD script

In [ ]:
open("prod.py","w").write('''
import os, sys
from common import *
TOTAL_NS = float(sys.argv[1]) if len(sys.argv) > 1 else 300
total = int(TOTAL_NS*1000/0.004)
sim, prmtop, inpcrd = make_sim()
resume = os.path.exists("chk.chk")
if resume: sim.loadCheckpoint("chk.chk")
else: sim.loadState("equil.xml")
sim.reporters.append(DCDReporter("traj.dcd", 25000, append=resume))
sim.reporters.append(StateDataReporter("log.csv", 25000, step=True, time=True, potentialEnergy=True,
                     temperature=True, density=True, speed=True, append=resume))
sim.reporters.append(CheckpointReporter("chk.chk", 25000))
print("starting at step", sim.currentStep, flush=True)
sim.step(total - sim.currentStep)
print("DONE")
''')

Run a short production test

In [ ]:
!python prod.py 1

Inspect recent simulation log records

In [ ]:
import pandas as pd
print(pd.read_csv("log.csv").tail(3))

Run the requested production simulation

In [ ]:
!rm -f traj.dcd chk.chk log.csv
!python prod.py 20